In [ ]:
import os
import yaml
import sys
root_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(root_path)
import tensorflow as tf
import keras
import argparse
from src.madnet import MADNet
from src.model_layers import colorize_img
from src.preprocessing import StereoDatasetCreator
from src.losses_and_metrics import SSIMLoss
import matplotlib.pyplot as plt
from types import SimpleNamespace
from src.config_preprocessing import normalize_config
# from src.infer_preprocessing import run_infer

# Load configuration from YAML file
config_path = os.path.join(root_path, "configs/infer_config.yaml")
if not os.path.exists(config_path):
    raise FileNotFoundError(f"Config file not found: {config_path}")
with open(config_path, "r") as f:
    cfg = yaml.safe_load(f)

cfg = normalize_config(cfg)
args = SimpleNamespace(**cfg)

2025-09-24 17:08:58.548559: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-09-24 17:08:59.210253: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-09-24 17:09:00.907651: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [ ]:
if args.output_path is None and args.num_adapt != 0:
    raise ValueError("No output_path provided for adaptation."
                        "Either set num_adapt=0 or provide a path to output_path"
                        f"Provided args: output_path: {args.output_path}, num_adapt: {args.num_adapt}")

# Initialise the model
model = MADNet(
    input_shape=(args.height, args.width, 3),
    # weights=args.weights_path,
    # num_adapt_modules=args.num_adapt,
    # mad_mode=args.mad_mode,
    search_range=args.search_range
)
optimizer = keras.optimizers.Adam(learning_rate=args.lr)
model.compile(
    optimizer=optimizer,
    loss=SSIMLoss(),
    metrics=None,
    run_eagerly=False
)
# Get inferencing data
predict_dataset = StereoDatasetCreator(
    left_dir=args.left_dir,
    right_dir=args.right_dir,
    batch_size=args.batch_size,
    height=args.height,
    width=args.width,
    shuffle=False,
    disp_dir=None
)
predict_ds = predict_dataset()
# inference the dataset
disparities = model.predict(predict_ds, steps=args.steps)

/home/ubuntu24/repos/madnet-deep-stereo-with-keras/.venv/lib/python3.13/site-packages/keras/src/models/model.py:158: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  Layer.__init__(self, *args, **kwargs)
I0000 00:00:1758726542.665095  339744 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13551 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4070 Ti SUPER, pci bus id: 0000:01:00.0, compute capability: 8.9
2025-09-24 17:09:24.810305: I external/local_xla/xla/service/service.cc:163] XLA service 0x7f5e94008d90 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-09-24 17:09:24.810343: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce RTX 4070 Ti SUPER, Compute Capability 8.9
2025-09-24 17:09:24.957924: I tensorflow/compiler/mlir/tens

39/39 ━━━━━━━━━━━━━━━━━━━━ 32s 46ms/step


In [ ]:
# View disparity predictions
if args.show_pred:
    for i in range(disparities.shape[0]):
        plt.axis("off")
        plt.grid(visible=None)
        disp = keras.ops.expand_dims(disparities[i, :, :, :], axis=0)
        plt.imshow(colorize_img(disp, cmap='jet')[0])
        plt.show()

In [4]:
# save the checkpoint and saved_model if it was updated
if args.num_adapt != 0:
    model.save_weights(args.output_path)